# Market Analysis Agent

The Market Analysis Agent is an AI-powered research assistant that delivers structured, enterprise-grade market intelligence. Given a company name, product, or industry, it produces detailed reports covering market trends, competitor analysis, user behaviour, and strategic opportunities — with a focus on the Chinese market.

In [2]:
!pip install -q aixplain

## Add your API key

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

In [ ]:
AixplainAPIKey = "<YOUR_AIXPLAIN_API_KEY>"

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey

In [3]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the Market Analysis Agent?

The goal of this agent is to make market research fast, structured, and actionable. Instead of manually compiling reports, you describe the company, product, or sector you want to analyse and the agent returns a comprehensive, structured report.

## Agent Architecture

The agent uses one custom script tool:

- **Market Analysis Planner** — converts a free-form research request into a structured JSON brief that specifies analysis dimensions (market trends, competitors, user behaviour, strategic opportunities), the target market, and the depth of each section.

## Agent Workflow

1. **Parse the request** — extract the subject, market scope, and desired analysis depth.
2. **Structure the brief** — the planner tool returns a JSON outline with dimensions, key questions, and suggested metrics for each section.
3. **Generate the report** — the agent fills in each section using its knowledge and reasoning, following the brief's structure.
4. **Deliver** — returns a complete market analysis report in structured markdown.

In [4]:
def plan_market_analysis(subject: str, market: str = "China", depth: str = "standard") -> str:
    """
    Converts a market research request into a structured analysis brief.

    Parameters:
        subject (str): The company, product, service, or industry to analyse.
                       Examples: 'WeChat Pay', 'electric vehicles', 'ByteDance short-video'.
        market (str): The target market or region (default: 'China').
                      Examples: 'China', 'Southeast Asia', 'global'.
        depth (str): Analysis depth — 'quick' (overview), 'standard' (default), or 'deep' (comprehensive).

    Returns:
        str: A JSON-formatted analysis brief with dimensions, key questions, and metrics per section.
    """
    import json

    depth_config = {
        "quick":    {"sections_per_dimension": 2, "metrics_per_section": 2},
        "standard": {"sections_per_dimension": 3, "metrics_per_section": 3},
        "deep":     {"sections_per_dimension": 5, "metrics_per_section": 5},
    }
    cfg = depth_config.get(depth, depth_config["standard"])

    brief = {
        "subject": subject,
        "market": market,
        "depth": depth,
        "dimensions": [
            {
                "name": "Market Trends",
                "key_questions": [
                    f"What are the major growth drivers for {subject} in {market}?",
                    f"What regulatory or macro trends are shaping {subject}?",
                    f"What emerging technologies are disrupting {subject}?",
                    f"What is the market size and projected CAGR for {subject} in {market}?",
                    f"How has {subject} evolved over the last 3-5 years?",
                ][:cfg["sections_per_dimension"]],
                "suggested_metrics": [
                    "Market size (USD)", "YoY growth rate", "CAGR (3-year)",
                    "Regulatory impact score", "Technology adoption rate",
                ][:cfg["metrics_per_section"]],
            },
            {
                "name": "Competitor Analysis",
                "key_questions": [
                    f"Who are the top 5 players in {subject} in {market}?",
                    f"What are the competitive advantages of each major player?",
                    f"What is the market share distribution among competitors?",
                    f"What differentiation strategies are competitors using?",
                    f"Who are the emerging challengers or disruptors?",
                ][:cfg["sections_per_dimension"]],
                "suggested_metrics": [
                    "Market share (%)", "Revenue (USD)", "User/customer count",
                    "Net Promoter Score", "Funding raised (USD)",
                ][:cfg["metrics_per_section"]],
            },
            {
                "name": "User Behaviour",
                "key_questions": [
                    f"Who is the primary user segment for {subject} in {market}?",
                    f"What are the key pain points users are trying to solve?",
                    f"What channels do users use to discover and adopt {subject}?",
                    f"What are the usage frequency and engagement patterns?",
                    f"What factors drive retention and churn?",
                ][:cfg["sections_per_dimension"]],
                "suggested_metrics": [
                    "DAU/MAU ratio", "Average session duration", "Conversion rate",
                    "Churn rate", "Customer lifetime value (LTV)",
                ][:cfg["metrics_per_section"]],
            },
            {
                "name": "Strategic Opportunities",
                "key_questions": [
                    f"What unmet needs exist in {subject} in {market}?",
                    f"What partnerships or integrations could accelerate growth?",
                    f"What geographic or demographic segments are underserved?",
                    f"What product or feature gaps do competitors have?",
                    f"What M&A or investment targets are attractive in {subject}?",
                ][:cfg["sections_per_dimension"]],
                "suggested_metrics": [
                    "Addressable market size", "Entry barrier score", "Time-to-market estimate",
                    "ROI potential", "Risk score",
                ][:cfg["metrics_per_section"]],
            },
        ],
        "output_format": {
            "structure": "markdown",
            "include_executive_summary": True,
            "include_sources_note": True,
        },
    }

    return json.dumps(brief, ensure_ascii=False, indent=2)

In [5]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

planner_tool = aix.Tool(
    name="Market Analysis Planner",
    integration=SCRIPT_INTEGRATION_ID,
    config={
        "code": inspect.getsource(plan_market_analysis),
        "function_name": "plan_market_analysis",
    },
)
planner_tool.save()
print(f"Created: {planner_tool.name}")

Created: Market Analysis Planner


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, a single planner tool that structures the analysis brief.
* The **instructions** that guide the agent's research and reporting behaviour.

In [6]:
market_agent = aix.Agent(
    name="Market Analysis Agent",
    description="Delivers structured, enterprise-grade market intelligence reports covering trends, competitors, user behaviour, and strategic opportunities — with deep expertise in the Chinese market.",
    instructions="""
    You are an expert market research analyst specialising in the Chinese market, with deep knowledge
    of Chinese consumer behaviour, regulatory environment, major platforms (WeChat, Alipay, Douyin,
    Taobao, JD, Baidu, Meituan, Didi, etc.), and emerging tech sectors (AI, EVs, fintech, e-commerce,
    SaaS, gaming, healthtech).

    When a user asks for market research:
    1. Call the Market Analysis Planner tool with the subject, target market, and depth.
    2. Use the returned structured brief as your report outline.
    3. Write a complete market analysis report in markdown, covering every dimension in the brief:
       - Executive Summary (3-5 bullet points)
       - Market Trends
       - Competitor Analysis (use a comparison table where helpful)
       - User Behaviour
       - Strategic Opportunities
    4. For each section, answer the key questions from the brief and reference the suggested metrics.
    5. End with a Sources note listing the types of data sources that would validate these findings
       (e.g. iResearch, QuestMobile, Analysys, CNNIC, company earnings reports).

    Guidelines:
    - Use specific numbers, market sizes, growth rates, and company names where known.
    - For Chinese companies and products, include both the English and Chinese name on first mention.
    - Flag any areas of high uncertainty or rapid change.
    - Keep the tone professional and direct — this is enterprise research, not marketing copy.
    - If the user asks for a 'quick' overview, keep each section to 2-3 bullet points.
    - If the user asks for a 'deep' report, expand each section with sub-sections and data tables.
    """,
    tools=[planner_tool],
)
market_agent.save()
print(f"Agent created: {market_agent.id}")

Agent created: 69c45eb5228f69075764f6a1


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.

In [7]:
result = market_agent.run("Give me a market analysis of the short-video industry in China")

In [8]:
print(result.data.output)

Market Analysis of the Short-Video Industry in China

Executive Summary
The short-video industry in China has seen exponential growth, driven by increasing smartphone penetration and social media engagement.
Major players include Douyin (抖音), Kuaishou (快手), and Bilibili (哔哩哔哩), with significant market shares and unique competitive advantages.
User engagement is high, with daily active users (DAUs) reaching hundreds of millions, indicating strong retention and growth potential.
Emerging technologies such as AI and 5G are expected to further disrupt and enhance the industry.
Strategic opportunities exist in underserved demographics and potential partnerships to enhance content offerings.

Market Trends
Growth Drivers: The rise of mobile internet usage and the popularity of social media platforms are key drivers.
Regulatory Trends: Increased scrutiny on content moderation and data privacy is shaping the industry landscape.
Emerging Technologies: AI-driven content recommendations and 5G te

In [9]:
result2 = market_agent.run("Who are the top competitors in the Chinese EV market and what is their market share?")
print(result2.data.output)

Market Analysis Report: Top Competitors in the Chinese EV Market

Executive Summary
The Chinese EV market is experiencing rapid growth, driven by government policies and increasing consumer demand.
Major players include BYD (比亚迪), Tesla (特斯拉), NIO (蔚来), Xpeng (小鹏), and Li Auto (理想).
Market share distribution shows BYD leading with approximately 25%, followed by Tesla at around 15%.
Emerging technologies such as battery innovations and autonomous driving are reshaping the competitive landscape.
Strategic opportunities exist in underserved geographic segments and through potential partnerships.

Market Trends
Growth Drivers: Government incentives, environmental concerns, and advancements in battery technology.
Regulatory Trends: Stricter emissions regulations and support for EV infrastructure development.
Emerging Technologies: Innovations in battery efficiency and autonomous driving capabilities.
Market Size: The Chinese EV market is projected to reach USD 200 billion by 2025, with a CA

In [10]:
result3 = market_agent.run("Quick overview of China's fintech and mobile payments landscape")
print(result3.data.output)

Market Analysis Report: China's Fintech and Mobile Payments Landscape

Executive Summary
- China's fintech and mobile payments market is experiencing rapid growth, driven by technological advancements and increasing consumer adoption.
- Major players include Alipay (支付宝), WeChat Pay (微信支付), and JD Pay (京东支付), dominating the market with significant market shares.
- User behavior indicates a strong preference for mobile payments, with a focus on convenience and security.
- Strategic opportunities exist in addressing unmet needs and forming partnerships to enhance service offerings.

Market Trends
- Growth Drivers: The major growth drivers include the increasing smartphone penetration, the rise of e-commerce, and supportive government policies promoting digital finance.
- Regulatory Trends: Regulatory frameworks are evolving, with the government emphasizing data security and consumer protection, impacting how fintech companies operate.
- Market Size: The market size for mobile payments in

In [11]:
# Multi-turn: follow up on the previous result
result4 = market_agent.run(
    "Go deeper on the strategic opportunities section — what's the best entry point for a foreign fintech?",
    session_id=result3.data.session_id
)
print(result4.data.output)

Market Analysis Report: Strategic Opportunities for Foreign Fintech in China

Executive Summary
The Chinese fintech landscape presents significant opportunities for foreign companies, driven by rapid digital adoption and evolving consumer needs.
Key entry points include partnerships with local players, focusing on underserved demographics, and leveraging emerging technologies.
Regulatory compliance and understanding local consumer behavior are critical for successful market entry.

Market Trends
Growth Drivers: The increasing smartphone penetration and the rise of e-commerce are major growth drivers, with mobile payments projected to continue expanding.
Regulatory Trends: The Chinese government is enhancing regulations around data security and consumer protection, which foreign fintechs must navigate carefully.
Market Size: The fintech market in China is estimated to be around $1 trillion, with a projected CAGR of 15% over the next five years.

Competitor Analysis
Company Name       | 

# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.

In [12]:
market_agent.save()

Agent(path=asma-furniturewala/market-analysis-agent/aixplain)

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).